# Midas Design Guide — Steel Composite Girder Design

**Companion notebook** for the corresponding chapter of the MIDAS training
manual *Design Guide for midas Civil — AASHTO LRFD*. The guide itself is
proprietary and is **not reproduced here** — this notebook contains only
original code and AASHTO LRFD / MBE article citations.

**How this notebook is used.** Work through the steel-composite design
process with this notebook alongside: each step carries its AASHTO
background in original words, a live Python environment for exploratory
checks and quick validation, a direct interface to Midas Civil through its
API, and customization to ODOT practice (Grade 50W unpainted weathering
steel, ODOT deck and haunch conventions, BDM load assumptions).
The guide's steel chapter, unlike its PSC chapter, contains **no worked
numerical example** — it presents the check equations and the Midas dialog
fields that drive them. So this notebook supplies the missing example: a
realistic ODOT-style two-span composite plate girder, carried through
every limit state with numbers a designer can rerun and modify. Where a
number must come from the Midas model itself (staged analysis forces,
envelope results, design result tables), there is a `TODO(midas-api)`
marker — those cells get finished against the live Civil NX JSON API when
a steel composite model exists.

**Scope of this chapter:**

1. The demo girder — an ODOT-style two-span composite plate girder
2. Cross-section proportion limits (LRFD 6.10.2)
3. Plastic moment $M_p$ and yield moment $M_y$ (Appendix D6), section
   classification (6.10.6.2), and the $D_p \le 0.42D_t$ ductility rule
4. Composite section properties — bare steel / $n$ / $3n$ / cracked, and
   effective flange width (6.10.1.1, 4.6.2.6)
5. Distribution factors and force effects (4.6.2.2; Midas API)
6. Constructibility — the noncomposite girder under wet concrete (6.10.3)
7. Service II permanent-deformation limits (6.10.4)
8. Fatigue (6.6.1, 6.10.5)
9. Strength I flexure — compact positive (6.10.7), noncompact negative
   (6.10.8), and the Appendix A6 path
10. Web shear and tension-field action (6.10.9)
11. Shear connectors — fatigue pitch and strength totals (6.10.10)
12. Transverse and bearing stiffeners (6.10.11)
13. Midas design-variable mapping and design result tables
14. What changed between editions — the guide's 4th-vs-6th list, extended
    to the current 9th

**What the `check()`s compare.** Unless a cell says otherwise, a `check()`
validates one Python derivation against another — the cell's hand calc vs
civilpy's implementation — or against a published constant (an AASHTO
limit, an AISC/ODOT table value). That proves the walkthrough and the
library agree on the article's math; it is *not* a software cross-check.
Labels mentioning Midas mean one of two things: a **Midas convention
computed here** (e.g. the D6.1 PNA-case table or the $M_y$ staged-modulus
iteration exactly as the guide documents them, evaluated in Python for
comparison), or a **Midas model value** pulled from Civil NX over the API
— live when connected, otherwise recorded values noted in the cell.
Midas's *design-module* result tables are not API-reachable, so
civilpy-vs-Midas design-check comparisons remain manual side-by-sides.

**Guide editions.** This notebook was written against the **2020 edition**
of the guide; the PDF archived in `snbi_ui/utils/manuals/Midas/` (from the
Midas support KB) is the **2014 edition**, which teaches this chapter to
AASHTO LRFD **4th (2007) and 6th (2012)** side by side. Unlike the
concrete articles (renumbered wholesale in 2017), the steel articles have
kept their numbers: 6.10.x (I-sections), 6.11.x (box/tub), and the
Appendix A6 / B6 / D6 machinery cited here read the same in the 4th
edition and the current **9th (2020)**. What changed between editions is
mostly *content* inside stable article numbers — the guide's own
"Difference Between AASHTO-LRFD 4th and 6th" section lists the 2007→2012
deltas, and the closing section of this notebook extends that list
through the 9th. Known guide-vs-current disagreements are flagged in
place with ⚠️ callouts.

In [ ]:
import math
import pandas as pd

# civilpy is installed editable into this env (pip install -e .)
from civilpy.structural.midas import MidasCivil, parse_result_table, envelope
from civilpy.structural.aashto.lrfd import (
    concrete, prestressed, steel, composite, distribution, lrfr,
    creep_shrinkage, appendix_a6, appendix_b6, appendix_d6,
)

# --- Midas Civil NX connection -------------------------------------------------
# The API is not running on this machine right now. Everything below that needs
# the live model is guarded by MIDAS_ONLINE and marked TODO(midas-api).
try:
    midas = MidasCivil()
    MIDAS_ONLINE = midas.ping()
except Exception:
    midas, MIDAS_ONLINE = None, False
print("Midas Civil NX online:", MIDAS_ONLINE)

In [ ]:
# --- Validation harness --------------------------------------------------------
# Every comparison in this notebook goes through check() so the end-of-notebook
# summary shows guide value vs civilpy value side by side.
RESULTS = []

def check(label, guide_value, civilpy_value, tol=0.01, unit=""):
    """Compare a guide-reported value against the civilpy-computed one.

    tol is relative (1% default) — the guide rounds intermediate values, so
    small drift is expected; flag anything beyond tol for investigation.
    """
    if guide_value is None or civilpy_value is None:
        status = "PENDING"
        diff = None
    else:
        diff = abs(civilpy_value - guide_value) / (abs(guide_value) or 1.0)
        status = "OK" if diff <= tol else "MISMATCH"
    RESULTS.append({"check": label, "guide": guide_value, "civilpy": civilpy_value,
                    "rel diff": diff, "unit": unit, "status": status})
    print(f"[{status}] {label}: guide={guide_value} civilpy={civilpy_value} {unit}")
    return status == "OK"

def summary():
    df = pd.DataFrame(RESULTS)
    if len(df):
        n_ok = (df.status == "OK").sum()
        print(f"{n_ok}/{len(df)} checks OK, "
              f"{(df.status == 'MISMATCH').sum()} mismatches, "
              f"{(df.status == 'PENDING').sum()} pending")
    return df

## 1. The demo girder — an ODOT-style two-span plate girder

The guide presents this chapter's checks abstractly, so the notebook runs
them on a concrete example sized the way an Ohio designer would:

| Item | Value | ODOT practice note |
|---|---|---|
| Spans | 2 × 120 ft continuous | two-span exercises *both* flexure signs |
| Girders | 4 @ 9'-0" spacing | interior girder designed here |
| Deck | 8.5 in structural | ODOT standard composite deck, $f'_c$ = 4.5 ksi (Class QC2) |
| Haunch | 2 in (top of top flange → bottom of deck) | ODOT convention; concrete over the flange only |
| Steel | Grade 50W, $F_y$ = 50 ksi, $F_u$ = 70 ksi | unpainted weathering steel — ODOT's default for new girders |
| Field section | TF 16×1, web 54×½, BF 18×1⅜ | positive-flexure region |
| Pier section | TF 18×1½, web 54×⁹⁄₁₆, BF 20×1¾ | negative-flexure region |

Why these plates: a 54-in web puts the steel depth near $0.033L$ —
comfortably above the $0.027L$ steel-depth minimum of Table 2.5.2.6.3-1
for continuous composite I-girders — and a two-plate schedule (field +
pier) is the smallest realistic set that shows how the checks change sign.
The bottom flange is wider and thicker than the top in *both* regions:
in positive flexure the deck does the compression work so the top flange
mainly needs constructibility stiffness; in negative flexure the bottom
flange is the discretely braced compression flange and earns its area.

The two cross sections are also mirrored as `GirderSide` objects — the
same dataclass civilpy's bolted-field-splice and composite-section
machinery consume — so every later section can pull geometry from one
place.

In [ ]:
# The single source of truth for the demo bridge. Everything downstream
# reads from DEMO / FIELD / PIER — change a plate here and rerun.
from civilpy.structural.aashto.lrfd.bolted_field_splice import Flange, GirderSide

DEMO = dict(
    spans_ft=[120.0, 120.0],       # two-span continuous
    girder_spacing_ft=9.0,
    n_girders=4,
    deck_thickness_in=8.5,         # structural thickness
    haunch_in=2.0,                 # top of top flange -> bottom of deck
    fc_deck_ksi=4.5,               # ODOT Class QC2
    fy_ksi=50.0,                   # Grade 50W
    fu_ksi=70.0,
    unit_weight_conc_kcf=0.150,
)

# plate schedule: (b_tf x t_tf, D x t_w, b_bf x t_bf), inches
FIELD = GirderSide(
    top_flange=Flange("Grade 50W", thickness=1.0, width=16.0),
    bottom_flange=Flange("Grade 50W", thickness=1.375, width=18.0),
    web_material="Grade 50W", web_thickness=0.5, web_depth=54.0,
    haunch=DEMO["haunch_in"],
)
PIER = GirderSide(
    top_flange=Flange("Grade 50W", thickness=1.5, width=18.0),
    bottom_flange=Flange("Grade 50W", thickness=1.75, width=20.0),
    web_material="Grade 50W", web_thickness=0.5625, web_depth=54.0,
    haunch=DEMO["haunch_in"],
)

def steel_summary(side, label):
    a = (side.top_flange.area + side.bottom_flange.area
         + side.web_thickness * side.web_depth)
    d = side.top_flange.thickness + side.web_depth + side.bottom_flange.thickness
    return dict(section=label, A_steel_in2=a, depth_in=d,
                weight_plf=round(a * 3.4, 1))

df_sections = pd.DataFrame([steel_summary(FIELD, "field (positive)"),
                            steel_summary(PIER, "pier (negative)")])

# steel depth vs the Table 2.5.2.6.3-1 minimum for continuous composite
# I-girders (0.027L) -- a proportioning sanity check, not a strength check
L = DEMO["spans_ft"][0]
d_steel = df_sections.depth_in.min()
print(f"steel depth {d_steel:.2f} in vs 0.027L = {0.027 * L * 12:.2f} in minimum")
check("steel depth >= 0.027L (Table 2.5.2.6.3-1) [published limit]",
      1.0, float(d_steel >= 0.027 * L * 12.0))
df_sections

## 2. Cross-section proportion limits — LRFD 6.10.2

Before any strength math, 6.10.2 screens the raw plate proportions. These
limits are not strength checks — they are the *applicability boundary* of
everything that follows: the flexure and shear resistances of 6.10.7–6.10.9
were calibrated on girders inside these ratios, and Midas reports a plain
"NG" when a section steps outside them. What each limit is protecting:

- **Web slenderness** $D/t_w \le 150$ without longitudinal stiffeners
  (6.10.2.1.1-1), $\le 300$ with them (6.10.2.1.2-1). Keeps the panel
  handleable in the shop and in lifting, and keeps elastic web
  bend-buckling from governing so early that the section is uneconomical.
- **Flange slenderness** $b_f/2t_f \le 12.0$ (6.10.2.2-1). A stockiness
  cap so the flange can be welded, gripped, and erected without folding,
  and so flange local buckling stays in the range the 6.10.8.2.2 curve
  covers.
- **Flange width** $b_f \ge D/6$ (6.10.2.2-2). The lateral-torsional
  buckling equations model the girder as flanges restraining a web; a
  too-narrow flange breaks that model.
- **Flange thickness** $t_f \ge 1.1t_w$ (6.10.2.2-3). The web
  bend-buckling coefficient assumes the flanges act as rotational
  restraints at the web's edges — the flange must be meaningfully stiffer
  than the plate it is bracing.
- **Flange inertia ratio** $0.1 \le I_{yc}/I_{yt} \le 10$ (6.10.2.2-4).
  Outside this band the section behaves like a tee, and the LTB
  derivations (which assume something I-shaped) stop applying.

> ⚠️ **Guide-vs-current note.** The guide presents these tables for the
> 4th/6th editions; the limits are unchanged in the 9th (2020) — same
> numbers, same article layout — so its table transfers cleanly. For
> box/tub sections the parallel limits live in 6.11.2 (webs may be
> inclined; the web limit applies along the slope).

> ⚠️ **Erratum (2014 edition, Tables 2.5 and 2.8).** The guide's web
> proportion tables print the two rows **swapped**: they show
> $D/t_w \le 150$ for webs *with* longitudinal stiffeners and $\le 300$
> for webs *without*. AASHTO has it the other way — 6.10.2.1.1-1 gives
> 150 for **un**stiffened webs, 6.10.2.1.2-1 gives 300 *with*
> longitudinal stiffeners (verified against the archived PDF at 150 dpi;
> check the 2020 printing against your copy). Read literally, the guide
> row would let an unstiffened web run to $D/t_w = 300$.

A practical companion rule worth knowing at this stage: C6.10.3.4.1
suggests $b_{fc} \ge L_{ship}/85$ for the *shipped field piece* so girders
survive handling before the deck exists — with 120-ft spans and a field
splice near the 0.7 point, a ~90-ft piece wants a compression flange of
at least ~12.7 in. Both demo flanges clear it.

In [ ]:
# Hand-calc every 6.10.2 ratio, then confirm civilpy's verdicts match.
rows = []
for label, side in (("field", FIELD), ("pier", PIER)):
    D, tw = side.web_depth, side.web_thickness
    for fl_label, fl in (("top", side.top_flange), ("bottom", side.bottom_flange)):
        rows.append(dict(section=label, flange=fl_label,
                         ratio="bf/2tf", value=fl.width / (2 * fl.thickness),
                         limit="<= 12", ok=fl.width / (2 * fl.thickness) <= 12))
        rows.append(dict(section=label, flange=fl_label,
                         ratio="bf >= D/6", value=fl.width,
                         limit=f">= {D/6:.1f}", ok=fl.width >= D / 6))
        rows.append(dict(section=label, flange=fl_label,
                         ratio="tf >= 1.1tw", value=fl.thickness,
                         limit=f">= {1.1*tw:.3f}", ok=fl.thickness >= 1.1 * tw))
    rows.append(dict(section=label, flange="-", ratio="D/tw",
                     value=D / tw, limit="<= 150", ok=D / tw <= 150))
    i_yt = side.bottom_flange.thickness * side.bottom_flange.width ** 3 / 12
    i_yc = side.top_flange.thickness * side.top_flange.width ** 3 / 12
    rows.append(dict(section=label, flange="-", ratio="Iyc/Iyt",
                     value=i_yc / i_yt, limit="0.1 - 10",
                     ok=0.1 <= i_yc / i_yt <= 10))
df_prop = pd.DataFrame(rows)
display(df_prop)

# civilpy's one-call screen per region (positive flexure: top flange is
# the compression flange; the function is orientation-agnostic for these
# geometric limits) -- hand calc vs civilpy
for label, side in (("field", FIELD), ("pier", PIER)):
    i_yc = side.top_flange.thickness * side.top_flange.width ** 3 / 12
    i_yt = side.bottom_flange.thickness * side.bottom_flange.width ** 3 / 12
    res = steel.proportion_limits(
        d_web=side.web_depth, t_w=side.web_thickness,
        b_fc=side.top_flange.width, t_fc=side.top_flange.thickness,
        b_ft=side.bottom_flange.width, t_ft=side.bottom_flange.thickness,
        i_yc=i_yc, i_yt=i_yt)
    hand_ok = df_prop[df_prop.section == label].ok.all()
    check(f"6.10.2 all proportion limits, {label} section [hand calc]",
          float(hand_ok), res.capacity)

# shipping-piece flange guideline, C6.10.3.4.1 [published rule of thumb]
L_ship_ft = 0.75 * DEMO["spans_ft"][0]     # field piece to the splice
b_min_ship = L_ship_ft * 12 / 85
print(f"shipped-piece minimum flange: {b_min_ship:.1f} in "
      f"(field top flange = {FIELD.top_flange.width} in)")
check("C6.10.3.4.1 bfc >= Lship/85, field top flange [published value]",
      1.0, float(FIELD.top_flange.width >= b_min_ship))

## 3. Composite section properties — LRFD 6.10.1.1, 4.6.2.6

A composite girder is really **three different cross sections wearing the
same plates**, and every stress check in this chapter sums over them
(6.10.1.1.1a–c):

| Section | Carries | Deck transformed by |
|---|---|---|
| Bare steel | DC1 — girder + wet deck + haunch + forms | — (no deck yet) |
| Long-term composite | DC2 + DW — parapets, future wearing surface | $3n$ |
| Short-term composite | LL + IM — transient loads | $n$ |

The $3n$ trick is how the spec handles creep without a time-step analysis
(C6.10.1.1.1b): concrete held under permanent stress relaxes, shedding
force to the steel, and tripling the modular ratio approximates the
50-plus-year outcome. Transient loads come and go before creep can act, so
live load rides on the stiffer $n$ section. Under **negative** flexure the
deck cracks; for strength the composite section is the steel plus the
longitudinal deck reinforcement only (6.10.1.1.1c), though 6.6.1.2.1 lets
fatigue-range and Service II stresses use the uncracked section where the
deck stays in net compression.

**Modular ratio.** $n = E_s/E_c$, and the steel chapter's commentary
(C6.10.1.1.1b) both tabulates rounded values ($f'_c$ = 3.6–4.6 ksi
$\rightarrow n = 8$) and blesses using them. For the demo deck's 4.5-ksi
concrete the notebook uses **n = 8**, matching both the commentary table
and long-standing Ohio line-girder practice.

> ⚠️ **Edition note — $E_c$ itself moved.** In the guide's era (4th–6th
> editions) $E_c = 33{,}000\,K_1 w_c^{1.5}\sqrt{f'_c}$ (5.4.2.4); the
> 8th edition (2017) replaced it with
> $E_c = 120{,}000\,K_1 w_c^{2.0} f_c'^{0.33}$, which the 9th keeps. For
> 0.145-kcf concrete at 4.5 ksi the two land within a few percent, and the
> commentary $n$-table is unchanged — but a hand check that hard-codes the
> old formula against a current-spec tool will drift.

**Effective flange width.** Since the 2008 interims (so: *after* the
guide's 4th-edition baseline), 4.6.2.6.1 for an interior girder is simply
the **tributary width** — the girder spacing.

> ⚠️ **Edition note.** The 4th edition computed interior effective width
> as $\min(L_{eff}/4,\ 12t_s + \max(t_w, b_f/2),\ S)$. The cell below
> evaluates both: for this demo's proportions the old rule also lands on
> the spacing, which is typical — the simplification rarely changed
> girder-bridge answers, which is why AASHTO made it.

In [ ]:
# --- modular ratio, both eras, vs civilpy -------------------------------
import numpy as np

fc = DEMO["fc_deck_ksi"]; wc = 0.145; Es = 29000.0
Ec_old = 33000.0 * wc ** 1.5 * math.sqrt(fc)          # 5.4.2.4, pre-2017
Ec_new = 120000.0 * wc ** 2.0 * fc ** 0.33            # 5.4.2.4, 8th/9th ed
Ec_c6  = 1820.0 * math.sqrt(fc)                       # C6.10.1.1.1b shortcut
print(f"Ec: old 33000w^1.5√fc = {Ec_old:.0f} ksi | 9th-ed 120000w^2fc^0.33 "
      f"= {Ec_new:.0f} ksi | C6.10.1.1.1b 1820√fc = {Ec_c6:.0f} ksi")
check("n = Es/Ec via C6.10.1.1.1b shortcut [hand calc]",
      Es / Ec_c6, composite.modular_ratio(fc))
n = 8.0   # commentary-table value for f'c = 3.6-4.6 ksi; used from here on
print(f"n used downstream: {n} (C6.10.1.1.1b table; computed values "
      f"{Es/Ec_old:.2f} / {Es/Ec_new:.2f} / {Es/Ec_c6:.2f})")

# --- effective width, current rule vs 4th-edition rule ------------------
S_in = DEMO["girder_spacing_ft"] * 12.0
ts = DEMO["deck_thickness_in"]
b_eff = S_in                                   # 4.6.2.6.1, interior girder
L_eff = 0.7 * DEMO["spans_ft"][0] * 12.0       # pos.-region effective span
b_eff_4th = min(L_eff / 4.0, 12.0 * ts + max(FIELD.web_thickness,
                                             FIELD.top_flange.width / 2.0),
                S_in)
check("effective width: 9th-ed tributary vs 4th-ed 3-part rule "
      "[hand calc, edition demo]", b_eff, b_eff_4th, unit="in")

# --- the three sections, hand calc vs CompositeGirder -------------------
girder_pos = composite.CompositeGirder(
    FIELD, deck_t=ts, deck_weff=b_eff, deck_fc=fc, n=n)

# hand calc, bare-steel state: A, y_bar (from bottom of bottom flange), I
tft, tfb = FIELD.top_flange, FIELD.bottom_flange
D, tw = FIELD.web_depth, FIELD.web_thickness
parts = [  # (A, y_mid, I0)
    (tfb.area, tfb.thickness / 2, tfb.width * tfb.thickness ** 3 / 12),
    (D * tw, tfb.thickness + D / 2, tw * D ** 3 / 12),
    (tft.area, tfb.thickness + D + tft.thickness / 2,
     tft.width * tft.thickness ** 3 / 12),
]
A_h = sum(p[0] for p in parts)
y_h = sum(p[0] * p[1] for p in parts) / A_h
I_h = sum(p[2] + p[0] * (p[1] - y_h) ** 2 for p in parts)
st = girder_pos.props("steel")
check("bare-steel A [hand calc]", A_h, st.area, unit="in^2")
check("bare-steel y_NA [hand calc]", y_h, st.y_na, unit="in")
check("bare-steel I [hand calc]", I_h, st.inertia, unit="in^4")

# all three states, with steel-fiber section moduli
d_steel = tfb.thickness + D + tft.thickness
rows = []
for state, note in (("steel", "DC1"), ("3n", "DC2 + DW"), ("n", "LL + IM")):
    p = girder_pos.props(state)
    rows.append(dict(state=state, carries=note, A_tr=round(p.area, 1),
                     y_NA=round(p.y_na, 2), I=round(p.inertia, 0),
                     S_bot=round(p.inertia / p.y_na, 0),
                     S_top_steel=round(p.inertia / abs(d_steel - p.y_na), 0)))
df_props = pd.DataFrame(rows)
df_props

In [ ]:
# --- minimum longitudinal deck reinforcement, 6.10.1.7 ------------------
# Wherever the deck sees tension under Service II, AASHTO wants at least
# 1% of the total deck cross-sectional area as longitudinal bars, no
# larger than #6, spaced <= 12 in, fy >= 60 ksi, with 2/3 of it in the
# top layer.  (The guide's Eq. 2.6 states the same 1% rule.)
A_deck = ts * b_eff
A_rs_req = 0.01 * A_deck
bar5 = 0.31  # in^2, #5 bar
n_top = math.ceil(b_eff / 5.5)     # #5 @ 5.5 in, top layer
n_bot = math.ceil(b_eff / 10.0)    # #5 @ 10 in, bottom layer
A_top, A_bot = n_top * bar5, n_bot * bar5
print(f"A_deck = {A_deck:.0f} in^2 -> need {A_rs_req:.2f} in^2; provided "
      f"top #5@5.5 = {A_top:.2f} + bottom #5@10 = {A_bot:.2f} "
      f"= {A_top + A_bot:.2f} in^2")
check("6.10.1.7 A_rs >= 1% A_deck [published limit]",
      1.0, float(A_top + A_bot >= A_rs_req))
check("6.10.1.7 two-thirds in top layer [published limit]",
      1.0, float(A_top >= (2.0 / 3.0) * A_rs_req))

# bar-center depths from the top of slab: 2.5 in top clear cover (deck
# surface with deicing salts, Table 5.10.1-1) and 1 in bottom cover
c_rt = 2.5 + 0.3125
c_rb = ts - 1.0 - 0.3125
A_rs = A_top + A_bot
c_lump = (A_top * c_rt + A_bot * c_rb) / A_rs
print(f"layer centers from slab top: c_rt = {c_rt:.2f} in, "
      f"c_rb = {c_rb:.2f} in, lumped centroid {c_lump:.2f} in")

# --- cracked (negative) composite section at the pier -------------------
girder_neg = composite.CompositeGirder(
    PIER, deck_t=ts, deck_weff=b_eff, deck_fc=fc, n=n,
    rebar_area=A_rs, rebar_cover=c_lump)

# hand calc of the cracked section: steel parts + lumped rebar
tft, tfb = PIER.top_flange, PIER.bottom_flange
D, tw = PIER.web_depth, PIER.web_thickness
d_pier = tfb.thickness + D + tft.thickness
y_bars = d_pier + DEMO["haunch_in"] + ts - c_lump
parts = [
    (tfb.area, tfb.thickness / 2, tfb.width * tfb.thickness ** 3 / 12),
    (D * tw, tfb.thickness + D / 2, tw * D ** 3 / 12),
    (tft.area, tfb.thickness + D + tft.thickness / 2,
     tft.width * tft.thickness ** 3 / 12),
    (A_rs, y_bars, 0.0),
]
A_h = sum(p[0] for p in parts)
y_h = sum(p[0] * p[1] for p in parts) / A_h
I_h = sum(p[2] + p[0] * (p[1] - y_h) ** 2 for p in parts)
cr = girder_neg.props("negative")
check("cracked pier section y_NA [hand calc]", y_h, cr.y_na, unit="in")
check("cracked pier section I [hand calc]", I_h, cr.inertia, unit="in^4")

rows = []
for state, note in (("steel", "DC1"), ("3n", "DC2+DW if uncracked"),
                    ("n", "LL+IM if uncracked"), ("negative", "cracked: strength")):
    p = girder_neg.props(state)
    rows.append(dict(state=state, carries=note, A_tr=round(p.area, 1),
                     y_NA=round(p.y_na, 2), I=round(p.inertia, 0),
                     S_bot=round(p.inertia / p.y_na, 0),
                     S_top_steel=round(p.inertia / abs(d_pier - p.y_na), 0)))
pd.DataFrame(rows)

## 4. Plastic moment and yield moment — Appendix D6

$M_p$ and $M_y$ are the two anchors every flexural check in this chapter
hangs from: section classification (6.10.6.2) asks how much web is in
compression *at the plastic moment*; the compact positive resistance
(6.10.7.1.2) is $M_p$ with a ductility penalty; the Appendix A6 negative
path interpolates between $M_p$ and yield via the web plastification
factors. Midas computes both silently for every design cell — this section
does the same math in the open.

**Plastic moment.** Set every steel component to $F_y$, the deck to a
uniform $0.85f'_c$ block, find the axis where compression balances
tension (the PNA), and take moments. Table D6.1-1 closes this into seven
cases by *where the PNA lands* — web, top flange, or five positions
through the deck — each with a closed-form $\bar{Y}$ and $M_p$. The case
conditions are just cumulative force totals: keep adding components to
the tension side until they outweigh what remains above.

**Ductility.** A compact composite section is allowed to reach $M_p$ only
if the PNA stays high: $D_p \le 0.42D_t$ (6.10.7.3). This is a crushing
guard — push the PNA deep and the deck's compression strain at the crest
of the plastic distribution outruns what concrete can deliver before the
steel finishes yielding.

**Yield moment.** For a composite girder $M_y$ is *not* $F_yS$, because
the section that resists the first slice of moment is not the section
that resists the last: DC1 stresses the bare steel, DC2 + DW the $3n$
section, and only the remainder $M_{AD}$ rides the short-term $n$ section
(D6.2.2). $M_y$ is the staged sum $M_{D1} + M_{D2} + M_{AD}$ for
whichever flange yields first. The same accounting explains most
"my hand calc doesn't match the program" tickets — a one-section
$F_yS$ estimate can miss by 20%.

> ⚠️ **Erratum (2014 edition, Table 2.7, Case I).** In the PNA-in-web row
> the guide's $M_p$ bracket prints $P_w d_w$ where AASHTO Table D6.1-1
> has $P_c d_c$ — the web is already accounted for by the leading
> $\frac{P_w}{2D}[\bar{Y}^2 + (D-\bar{Y})^2]$ term, so the printed row
> double-counts the web and drops the compression flange entirely. (The
> same row also writes $(t-\bar{Y})^2$ for $(D-\bar{Y})^2$.) The cell
> below implements AASHTO's table; the hand calc proves the demo numbers
> against an independent force-by-force sum.

> ⚠️ **Guide caution (Figs. 2.2 / 2.4, section classification).** The
> flowcharts test $\min(F_{yc}, F_{yt}) \le 70$ ksi. AASHTO 6.10.6.2.2
> and 6.10.6.2.3 require the yield strength of **each** flange (and the
> web) to satisfy the limit — for a hybrid girder with one flange over
> 70 ksi the guide's min() test passes a section the spec excludes.

In [ ]:
# --- plastic moment, field section: hand calc vs civilpy ----------------
from civilpy.structural.aashto.lrfd import plastic_moment

# plastic component forces (kip) — the Table D6.1-1 vocabulary
Ps = 0.85 * fc * b_eff * ts
Pc = 50.0 * FIELD.top_flange.width * FIELD.top_flange.thickness
Pw = 50.0 * FIELD.web_depth * FIELD.web_thickness
Pt = 50.0 * FIELD.bottom_flange.width * FIELD.bottom_flange.thickness
print(f"Ps={Ps:.1f}  Pc={Pc:.1f}  Pw={Pw:.1f}  Pt={Pt:.1f} kip")

# case walk (no deck rebar in positive flexure — its effect is ~1%):
#   I:  Pt+Pw >= Pc+Ps ?     II:  Pt+Pw+Pc >= Ps ?     else deck cases
print(f"case I  ? Pt+Pw = {Pt+Pw:.0f} >= Pc+Ps = {Pc+Ps:.0f} : {Pt+Pw >= Pc+Ps}")
print(f"case II ? Pt+Pw+Pc = {Pt+Pw+Pc:.0f} >= Ps = {Ps:.0f} : {Pt+Pw+Pc >= Ps}")
# -> neither: the PNA is in the deck (case III with no rebar terms)

# case III hand calc: Ybar = ts*(Pc+Pw+Pt)/Ps, slab block above the PNA
Ybar_h = ts * (Pc + Pw + Pt) / Ps
th = DEMO["haunch_in"]
y_c = ts + th + FIELD.top_flange.thickness / 2
y_w = ts + th + FIELD.top_flange.thickness + FIELD.web_depth / 2
y_t = (ts + th + FIELD.top_flange.thickness + FIELD.web_depth
       + FIELD.bottom_flange.thickness / 2)
Mp_h = (Ybar_h ** 2 * Ps / (2 * ts)
        + Pc * (y_c - Ybar_h) + Pw * (y_w - Ybar_h) + Pt * (y_t - Ybar_h))

pm_pos = plastic_moment(
    flexure="positive",
    b_c=FIELD.top_flange.width, t_c=FIELD.top_flange.thickness, f_yc=50.0,
    d_web=FIELD.web_depth, t_w=FIELD.web_thickness, f_yw=50.0,
    b_t=FIELD.bottom_flange.width, t_t=FIELD.bottom_flange.thickness,
    f_yt=50.0, f_c=fc, b_s=b_eff, t_s=ts, t_haunch=th)
print(f"civilpy: case {pm_pos.case} ({pm_pos.pna}), "
      f"Ybar = {pm_pos.y_bar:.3f} in, Mp = {pm_pos.mp/12:.0f} k-ft")
check("D6.1 case III Ybar, field section [hand calc]",
      Ybar_h, pm_pos.y_bar, unit="in")
check("D6.1 Mp, field section [hand calc]", Mp_h, pm_pos.mp, unit="k-in")

# ductility, 6.10.7.3: Dp = PNA depth from top of deck, Dt = total depth
Dp, Dt_tot = pm_pos.dp, pm_pos.dt
print(f"Dp/Dt = {Dp:.2f}/{Dt_tot:.2f} = {Dp/Dt_tot:.3f}  (limit 0.42)")
check("6.10.7.3 ductility Dp <= 0.42 Dt [published limit]",
      1.0, float(Dp <= 0.42 * Dt_tot))

# pier section, negative flexure (Table D6.1-2) — rebar carries the deck
pm_neg = plastic_moment(
    flexure="negative",
    b_c=PIER.bottom_flange.width, t_c=PIER.bottom_flange.thickness,
    f_yc=50.0,
    d_web=PIER.web_depth, t_w=PIER.web_thickness, f_yw=50.0,
    b_t=PIER.top_flange.width, t_t=PIER.top_flange.thickness, f_yt=50.0,
    t_s=ts, t_haunch=th,
    a_rt=A_top, c_rt=c_rt, a_rb=A_bot, c_rb=c_rb, f_yr=60.0)
print(f"pier: case {pm_neg.case} ({pm_neg.pna}), Mp = {pm_neg.mp/12:.0f} "
      f"k-ft, Dcp = {pm_neg.dcp:.2f} in")

# hand check of the negative case-I bracket — the erratum row, done right
Prt, Prb = 60.0 * A_top, 60.0 * A_bot
Pc_n = 50.0 * PIER.bottom_flange.width * PIER.bottom_flange.thickness
Pw_n = 50.0 * PIER.web_depth * PIER.web_thickness
Pt_n = 50.0 * PIER.top_flange.width * PIER.top_flange.thickness
D = PIER.web_depth
Yb = (D / 2) * ((Pc_n - Pt_n - Prt - Prb) / Pw_n + 1)
y_pna = ts + th + PIER.top_flange.thickness + Yb
Mp_n = (Pw_n / (2 * D)) * (Yb ** 2 + (D - Yb) ** 2) + (
    Prt * (y_pna - c_rt) + Prb * (y_pna - c_rb)
    + Pt_n * (y_pna - (ts + th + PIER.top_flange.thickness / 2))
    + Pc_n * ((ts + th + PIER.top_flange.thickness + D
               + PIER.bottom_flange.thickness / 2) - y_pna))
check("D6.1-2 case I Mp, pier section [hand calc]",
      Mp_n, pm_neg.mp, unit="k-in")

In [ ]:
# --- where the PNA lands: both regions, to scale ------------------------
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 6), sharey=True)
for ax, side, pm, title in (
        (axes[0], FIELD, pm_pos, "field section — positive flexure"),
        (axes[1], PIER, pm_neg, "pier section — negative flexure")):
    th = DEMO["haunch_in"]
    # drawn as built: the physical top flange up top in both regions
    t_top, b_top = side.top_flange.thickness, side.top_flange.width
    t_bot, b_bot = side.bottom_flange.thickness, side.bottom_flange.width
    D, tw = side.web_depth, side.web_thickness
    half = b_eff / 2

    # deck: shaded when it carries compression, dotted outline when cracked
    deck_fc = dict(facecolor="0.85", edgecolor="k")
    if pm.flexure == "negative":
        deck_fc = dict(facecolor="none", edgecolor="k", linestyle=":")
    ax.add_patch(plt.Rectangle((-half, 0), b_eff, ts, **deck_fc))
    y0 = ts + th
    ax.add_patch(plt.Rectangle((-b_top / 2, y0), b_top, t_top,
                               facecolor="0.55", edgecolor="k"))
    ax.add_patch(plt.Rectangle((-tw / 2, y0 + t_top), tw, D,
                               facecolor="0.55", edgecolor="k"))
    ax.add_patch(plt.Rectangle((-b_bot / 2, y0 + t_top + D), b_bot, t_bot,
                               facecolor="0.55", edgecolor="k"))
    # rebar dots for the cracked pier section
    if pm.flexure == "negative":
        for c_layer, n_dots in ((c_rt, 12), (c_rb, 7)):
            xs = [(-half + 4) + k * (b_eff - 8) / (n_dots - 1)
                  for k in range(n_dots)]
            ax.plot(xs, [c_layer] * n_dots, "ko", ms=3)

    ax.axhline(pm.y_pna, color="crimson", lw=2, linestyle="--")
    ax.annotate(f"PNA — case {pm.case}\n({pm.pna})",
                xy=(half * 0.35, pm.y_pna), xytext=(half * 0.35,
                pm.y_pna + 11), fontsize=9, color="crimson",
                arrowprops=dict(arrowstyle="->", color="crimson"))
    ax.text(0, -3.5, f"$M_p$ = {pm.mp/12:,.0f} k-ft", ha="center",
            fontsize=10)
    ax.set_title(title, fontsize=10)
    ax.set_xlim(-half - 6, half + 6)
    ax.set_ylim(ts + th + t_top + D + t_bot + 4, -8)
    ax.set_aspect("equal")
    ax.axis("off")
fig.suptitle("Plastic neutral axis location, Table D6.1 cases", y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# --- yield moment, D6.2.2: the staged sum --------------------------------
from civilpy.structural.aashto.lrfd import yield_moment_composite

# unfactored line-girder moments at the 0.375L positive-moment section of
# a prismatic two-span continuous girder (M+ = 9wL^2/128 = 0.0703 wL^2).
# These are hand statics on a prismatic idealization — a staged Midas
# model refines them.  TODO(midas-api): replace with construction-stage
# envelope results once a steel composite model exists.
w_dc1 = 1.40   # klf: girder+details ~0.25, deck 0.956, haunch, SIP forms
w_dc2 = 0.32   # klf: 2 parapets / 4 girders
w_dw = 0.54    # klf: 0.060 ksf future wearing surface x 9 ft
L = DEMO["spans_ft"][0]
coef = 9.0 / 128.0
M_D1 = coef * w_dc1 * L ** 2 * 12.0    # kip-in, on the bare steel
M_D2 = coef * (w_dc2 + w_dw) * L ** 2 * 12.0   # kip-in, on the 3n section
print(f"M_D1 = {M_D1/12:.0f} k-ft (steel), M_D2 = {M_D2/12:.0f} k-ft (3n)")

p_st = girder_pos.props("steel")
p_lt = girder_pos.props("3n")
p_n = girder_pos.props("n")
d_stl = FIELD.top_flange.thickness + FIELD.web_depth + FIELD.bottom_flange.thickness

def moduli(p):
    return p.inertia / abs(d_stl - p.y_na), p.inertia / p.y_na  # top, bot

s_nc_t, s_nc_b = moduli(p_st)
s_lt_t, s_lt_b = moduli(p_lt)
s_st_t, s_st_b = moduli(p_n)
ym = yield_moment_composite(
    m_d1=M_D1, m_d2=M_D2,
    s_nc_top=s_nc_t, s_nc_bot=s_nc_b, s_lt_top=s_lt_t, s_lt_bot=s_lt_b,
    s_st_top=s_st_t, s_st_bot=s_st_b, f_y_top=50.0, f_y_bot=50.0)
print(f"MAD: top {ym.m_ad_top/12:,.0f} / bottom {ym.m_ad_bot/12:,.0f} k-ft "
      f"-> My = {ym.my/12:,.0f} k-ft ({ym.governing_flange} flange governs)")

# hand check, bottom flange: MAD = S_st*(Fy - MD1/S_nc - MD2/S_lt)
MAD_b = s_st_b * (50.0 - M_D1 / s_nc_b - M_D2 / s_lt_b)
check("D6.2.2 M_AD bottom flange [hand calc]", MAD_b, ym.m_ad_bot,
      unit="k-in")
check("D6.2.2 My = MD1+MD2+MAD [hand calc]",
      M_D1 + M_D2 + MAD_b, ym.my, unit="k-in")

# the one-section shortcut, for contrast (why staged accounting matters):
My_naive = 50.0 * s_st_b
print(f"naive Fy*S_n(bot) = {My_naive/12:,.0f} k-ft vs staged "
      f"{ym.my/12:,.0f} k-ft ({(My_naive/ym.my - 1)*100:+.1f}%)")

# and the plastic-vs-yield spread the classification cares about
print(f"Mp/My = {pm_pos.mp/ym.my:.3f}")

In [ ]:
# --- section classification, 6.10.6.2 ------------------------------------
# positive flexure (6.10.6.2.2): straight bridge, Fy <= 70 both flanges,
# web within 6.10.2.1.1, and 2*Dcp/tw <= 3.76*sqrt(E/Fyc).  Our PNA is in
# the DECK, so Dcp = 0 and the slenderness test is trivially satisfied —
# the usual outcome for composite positive flexure.
lam_pw = 3.76 * math.sqrt(29000.0 / 50.0)
cls_pos = steel.classify_composite_positive(
    d_cp=pm_pos.dcp, t_w=FIELD.web_thickness, f_yc=50.0, f_yt=50.0,
    d_web=FIELD.web_depth)
print(cls_pos.details)
check("6.10.6.2.2 compact limit 3.76*sqrt(E/Fyc) [hand calc]",
      lam_pw, cls_pos.details["3.76sqrt(E/Fyc)"])
check("6.10.6.2.2 field section classifies compact [hand calc]",
      1.0, cls_pos.capacity)

# negative flexure (6.10.6.2.3): the slender-web screen decides between
# Appendix A6 (up to Mp) and 6.10.8 (stress-based).  Dc here is the
# cracked elastic section's web-compression depth (compression is at the
# bottom over the pier).
Dc_neg = cr.y_na - PIER.bottom_flange.thickness
i_yc_p = PIER.bottom_flange.thickness * PIER.bottom_flange.width ** 3 / 12
i_yt_p = PIER.top_flange.thickness * PIER.top_flange.width ** 3 / 12
cls_neg = steel.classify_web_negative(
    d_c=Dc_neg, t_w=PIER.web_thickness, f_yc=50.0, f_yt=50.0,
    i_yc=i_yc_p, i_yt=i_yt_p)
lam_rw = 5.7 * math.sqrt(29000.0 / 50.0)
print(f"2Dc/tw = {2*Dc_neg/PIER.web_thickness:.1f} vs "
      f"5.7*sqrt(E/Fyc) = {lam_rw:.1f} -> {cls_neg.details['classification']}")
check("6.10.6.2.3 2Dc/tw, pier section [hand calc]",
      2 * Dc_neg / PIER.web_thickness, cls_neg.details["2Dc/tw"])
check("6.10.6.2.3 pier web nonslender (A6 eligible) [hand calc]",
      1.0, cls_neg.capacity)

## 5. Distribution factors and force effects — LRFD 4.6.2.2, 3.6.1

**Why DFs exist here at all.** Midas analyzes the whole 3D bridge, so
when the guide's workflow is followed literally there is no distribution
factor anywhere — each girder simply gets what the grillage gives it.
The DF machinery below is the *line-girder* equivalent: one girder,
per-lane live load scaled by the Table 4.6.2.2 factors. Running both is
the standard QC sandwich — the hand line-girder numbers bound and
sanity-check the model, and any large disagreement means the model (or
the hand idealization) is wrong. That cross-check is exactly what this
notebook's `TODO(midas-api)` cells will do once a steel composite model
exists.

The interior-girder factors for a concrete deck on steel I-girders
(type (a) in Table 4.6.2.2.1-1) need one section-dependent quantity: the
longitudinal stiffness parameter $K_g = n(I + Ae_g^2)$ (4.6.2.2.1-1),
computed on the **noncomposite** steel with $e_g$ from girder centroid to
deck centroid. Everything else is geometry. Multiple presence is baked
into the formulas (C4.6.2.2.1) — never multiply it in again; the only
time it appears explicitly is the lever rule and the 1.2 divide-out for
fatigue's single-lane factor.

**Force effects.** The demo's dead loads ride two-span continuous
statics; live load comes from an influence-line sweep of the HL-93 trio
(design truck with its 14–30 ft variable rear axle, tandem, 0.64-klf
lane), with the 3.6.1.3.1 two-truck 90% case governing the pier negative
moment. Everything is computed in the open in the next cell — a
prismatic idealization, so the numbers are honest line-girder values a
staged nonprismatic model will refine, not match exactly.

In [ ]:
# --- Kg and the interior-girder DFs, hand vs civilpy ---------------------
p_st = girder_pos.props("steel")
d_stl = (FIELD.top_flange.thickness + FIELD.web_depth
         + FIELD.bottom_flange.thickness)
e_g = (d_stl + DEMO["haunch_in"] + ts / 2.0) - p_st.y_na
K_g = distribution.longitudinal_stiffness_kg(
    n_modular=n, i_girder=p_st.inertia, a_girder=p_st.area, e_g=e_g)
K_g_hand = n * (p_st.inertia + p_st.area * e_g ** 2)
check("4.6.2.2.1-1 Kg [hand calc]", K_g_hand, K_g, unit="in^4")
print(f"eg = {e_g:.2f} in, Kg = {K_g:,.0f} in^4")

S_ft, L_ft = DEMO["girder_spacing_ft"], DEMO["spans_ft"][0]
df_m = distribution.moment_df_interior(S_ft, L_ft, ts, K_g,
                                       n_beams=DEMO["n_girders"])
df_v = distribution.shear_df_interior(S_ft, L_ft, ts,
                                      n_beams=DEMO["n_girders"])
# hand check of the governing (multi-lane) moment formula
stiff = K_g / (12.0 * L_ft * ts ** 3)
dfm_hand = 0.075 + (S_ft / 9.5) ** 0.6 * (S_ft / L_ft) ** 0.2 * stiff ** 0.1
check("4.6.2.2.2b multi-lane moment DF [hand calc]",
      dfm_hand, df_m.multi_lane)
assert df_m.applicable and df_v.applicable
DF_M, DF_V = df_m.governing, df_v.governing
# fatigue: single-lane DF with the built-in 1.2 multiple presence removed
DF_FAT = df_m.one_lane / 1.2
print(f"DF moment {DF_M:.3f} | shear {DF_V:.3f} | fatigue {DF_FAT:.3f} "
      f"(lanes/girder)")

In [ ]:
# --- two-span influence lines and the HL-93 envelope ---------------------
# Closed-form continuous-beam kernel (EI constant, equal spans): the pier
# moment for a unit load, then superposition for any section in span 1.
# Prismatic idealization — TODO(midas-api): compare against the staged
# Civil NX envelopes when a steel composite model exists.
L_in_ft = L_ft

def m_pier_unit(s):
    """Pier moment M_B (k-ft/kip) for a unit load at global position s
    (ft, 0 at the left abutment, both spans)."""
    a = s if s <= L_in_ft else 2 * L_in_ft - s     # distance from the
    return -a * (L_in_ft ** 2 - a ** 2) / (4 * L_in_ft ** 2)  # near abutment

def m_section_unit(x, s):
    """Moment at section x of span 1 for a unit load at s: simple-span
    moment plus the linear end-moment correction M_B * x/L."""
    mb = m_pier_unit(s)
    m_simple = 0.0
    if s <= L_in_ft:
        a = s
        m_simple = x * (L_in_ft - a) / L_in_ft if x <= a \
            else a * (L_in_ft - x) / L_in_ft
    return m_simple + mb * x / L_in_ft

def v_pier_unit(s):
    """Shear just left of the pier for a unit load at s (kip/kip)."""
    mb = m_pier_unit(s)
    if s <= L_in_ft:
        r_a = (L_in_ft - s) / L_in_ft + mb / L_in_ft
        return r_a - 1.0
    return mb / L_in_ft

x_star = 0.375 * L_in_ft                     # positive-moment section, ft
s_grid = np.linspace(0.0, 2 * L_in_ft, 4801)
il_pos = np.array([m_section_unit(x_star, s) for s in s_grid])
il_pier = np.array([m_pier_unit(s) for s in s_grid])
il_v = np.array([v_pier_unit(s) for s in s_grid])

# sanity anchors: uniform load on both spans, closed form vs IL integral
w = 1.0
m_pier_exact = -w * L_in_ft ** 2 / 8.0
m_pier_il = w * np.trapezoid(il_pier, s_grid)
check("IL sanity: pier moment under uniform w [hand calc]",
      m_pier_exact, m_pier_il, unit="k-ft")
m_pos_exact = (3 * w * L_in_ft / 8) * x_star - w * x_star ** 2 / 2 \
    - 0.0  # R_A = 3wL/8 for both spans loaded
check("IL sanity: 0.375L moment under uniform w [hand calc]",
      m_pos_exact, w * np.trapezoid(il_pos, s_grid), unit="k-ft")

def axle_effect(il, axles, spacings):
    """Envelope one vehicle (axle loads kip, spacings ft) over the IL,
    both travel directions.  Vectorized: every IL endpoint is zero, so
    np.interp's clamping makes off-bridge axles contribute nothing."""
    offs = np.concatenate(([0.0], np.cumsum(spacings)))
    axles = np.asarray(axles, dtype=float)
    best_max, best_min = 0.0, 0.0
    for pattern in (offs, offs[::-1] - offs[-1]):
        pos = s_grid[:, None] - pattern[None, :]     # leads x axles
        vals = np.interp(pos, s_grid, il) @ axles
        best_max = max(best_max, float(vals.max()))
        best_min = min(best_min, float(vals.min()))
    return best_max, best_min

def lane_effect(il):
    pos = np.trapezoid(np.clip(il, 0, None), s_grid) * 0.64
    neg = np.trapezoid(np.clip(il, None, 0), s_grid) * 0.64
    return pos, neg

def hl93(il):
    """Per-lane HL-93 max/min (k-ft or kip) with IM on vehicles only."""
    truck = [8.0, 32.0, 32.0]
    tr_max, tr_min = zip(*[axle_effect(il, truck, [14.0, r])
                           for r in (14.0, 22.0, 30.0)])
    td_max, td_min = axle_effect(il, [25.0, 25.0], [4.0])
    ln_max, ln_min = lane_effect(il)
    pos = max(1.33 * max(tr_max), 1.33 * td_max) + ln_max
    neg = min(1.33 * min(tr_min), 1.33 * td_min) + ln_min
    # 3.6.1.3.1: negative moment between contraflexure points may instead
    # be 90% of TWO trucks (fixed 14-ft rears, >= 50 ft apart) + 90% lane
    two_tr = min(axle_effect(il, truck + truck,
                             [14.0, 14.0, gap, 14.0, 14.0])[1]
                 for gap in (50.0, 60.0, 70.0, 82.0, 90.0))
    neg = min(neg, 0.9 * (1.33 * two_tr) + 0.9 * ln_min)
    return pos, neg

M_pos_lane, _ = hl93(il_pos)
_, M_neg_lane = hl93(il_pier)
_, V_pier_lane = hl93(il_v)
print(f"per-lane HL-93 (+IM): M+ = {M_pos_lane:.0f} k-ft, "
      f"M-pier = {M_neg_lane:.0f} k-ft, V-pier = {V_pier_lane:.1f} kip")

# --- assemble the demo's unfactored + Strength I demands -----------------
coef_pos, coef_pier = 9.0 / 128.0, -1.0 / 8.0
def dead_moments(wq):
    return coef_pos * wq * L_ft ** 2, coef_pier * wq * L_ft ** 2

M1p, M1n = dead_moments(w_dc1)
M2p, M2n = dead_moments(w_dc2)
Mwp, Mwn = dead_moments(w_dw)
MLp = DF_M * M_pos_lane
MLn = DF_M * M_neg_lane
V1, V2, Vw = (5 * wq * L_ft / 8 for wq in (w_dc1, w_dc2, w_dw))
VL = DF_V * abs(V_pier_lane)

DEMANDS = dict(
    # unfactored, per interior girder, k-ft / kip
    m_dc1_pos=M1p, m_dc2_pos=M2p, m_dw_pos=Mwp, m_ll_pos=MLp,
    m_dc1_neg=M1n, m_dc2_neg=M2n, m_dw_neg=Mwn, m_ll_neg=MLn,
    v_dc1=V1, v_dc2=V2, v_dw=Vw, v_ll=VL,
    # Strength I (3.4.1): 1.25 DC + 1.5 DW + 1.75 (LL+IM)
    mu_pos=1.25 * (M1p + M2p) + 1.5 * Mwp + 1.75 * MLp,
    mu_neg=1.25 * (M1n + M2n) + 1.5 * Mwn + 1.75 * MLn,
    vu_pier=1.25 * (V1 + V2) + 1.5 * Vw + 1.75 * VL,
)
pd.DataFrame([
    dict(effect="M+ @ 0.375L (k-ft)", DC1=round(M1p), DC2=round(M2p),
         DW=round(Mwp), **{"LL+IM": round(MLp)},
         StrengthI=round(DEMANDS["mu_pos"])),
    dict(effect="M- @ pier (k-ft)", DC1=round(M1n), DC2=round(M2n),
         DW=round(Mwn), **{"LL+IM": round(MLn)},
         StrengthI=round(DEMANDS["mu_neg"])),
    dict(effect="V @ pier (kip)", DC1=round(V1, 1), DC2=round(V2, 1),
         DW=round(Vw, 1), **{"LL+IM": round(VL, 1)},
         StrengthI=round(DEMANDS["vu_pier"], 1)),
])

In [ ]:
if MIDAS_ONLINE:
    # Verified vs live Civil NX 2026-07-27: /post/TABLE selects by
    # TABLE_TYPE ("BEAMFORCE"); TABLE_NAME is just a label.
    try:
        resp = midas.result_table("BeamForce",
                                  table_type="BEAMFORCE")
        display(pd.DataFrame(parse_result_table(resp)).head())
    except Exception as err:
        # a fresh/unanalyzed session (or a DB edit, or a pre-mode view
        # switch) clears results — analyze and rerun this cell
        print("no results in the session:", str(err)[-80:])
else:
    print("Midas offline — skipping BeamForce (BEAMFORCE)")

## 6. Constructibility — LRFD 6.10.3

The girder's worst day is often the deck pour: the full wet-concrete
weight is on the **bare steel**, the top flange is in compression with
nothing but cross-frames bracing it, and none of it is recoverable —
6.10.3 requires the structure to stay elastic (no nominal yielding, no
reliance on post-buckling strength) through every casting stage, because
locked-in construction deformations never come out. Midas drives this
check from the construction-stage "Dead (Before Composite)" forces; here
the same condition is the 1.25·DC1 moment on the noncomposite section.

Three families of checks (6.10.3.2, applied per flange between brace
points, $L_b$ = 25 ft cross-frame spacing for the demo):

1. **Compression flange** (6.10.3.2.1): nominal yielding
   $f_{bu} + f_l \le \phi_f R_h F_{yc}$, flexural buckling
   $f_{bu} + \tfrac{1}{3}f_l \le \phi_f F_{nc}$ with $F_{nc}$ the
   smaller of local buckling (6.10.8.2.2) and LTB (6.10.8.2.3), and — for
   slender webs — bend-buckling $f_{bu} \le \phi_f F_{crw}$ (6.10.1.9.1).
2. **Tension flange** (6.10.3.2.2): $f_{bu} + f_l \le \phi_f R_h F_{yt}$.
3. **Web shear** (6.10.3.3): $V_u \le \phi_v V_{cr}$ — shear *buckling*
   only. Tension-field action is deliberately off the table during
   construction: the post-buckled diagonal field needs anchorage and
   accepts visible web deformation, neither of which belongs on a girder
   that still has a deck to receive.

**Where $f_l$ comes from** — the guide never says: deck-overhang
falsework brackets torquing the exterior girders (the classic source),
staggered cross-frames, skewed supports, and wind on the bare pair. For
this demo's *interior* girder with symmetric adjacent pours, $f_l
\approx 0$ and the equations below carry it symbolically; exterior
girders on this bridge would need the bracket force worked out (C6.10.3.4
gives the geometry) and $f_l \le 0.6F_{yf}$ per 6.10.1.6.

**The deck itself** (6.10.3.2.4): staged casting puts already-hardened
deck regions into tension when later pours load the span — wherever that
tension exceeds $\phi f_r$ (with $f_r = 0.24\sqrt{f'_c}$ and
$\phi = 0.9$), the 6.10.1.7 one-percent longitudinal reinforcement must
already be in place. This is why the pour sequence drawing matters:
positive-moment regions first, pier caps last, so the pier-region deck is
as late and as reinforced as possible.

In [ ]:
# --- casting-stage flexure on the bare steel -----------------------------
# 3.4.2.1: permanent-load factor for construction checks >= 1.25
gam_c = 1.25
M_cast = gam_c * DEMANDS["m_dc1_pos"] * 12.0     # kip-in at 0.375L
S_top_nc, S_bot_nc = s_nc_t, s_nc_b
f_bu_c = M_cast / S_top_nc      # top (compression) flange, wet deck
f_bu_t = M_cast / S_bot_nc      # bottom (tension) flange
f_l = 0.0                       # interior girder, symmetric pours
print(f"casting: 1.25*M_DC1 = {M_cast/12:.0f} k-ft -> fbu top "
      f"{f_bu_c:.1f} ksi / bottom {f_bu_t:.1f} ksi")

# noncomposite Dc: web in compression above the elastic NA
Dc_nc = d_stl - p_st.y_na - FIELD.top_flange.thickness
L_b = 25.0 * 12.0               # cross-frame spacing

# LTB (6.10.8.2.3) — hand rt/Lp/Lr against civilpy
bfc, tfc = FIELD.top_flange.width, FIELD.top_flange.thickness
rt_h = bfc / math.sqrt(12.0 * (1.0 + Dc_nc * FIELD.web_thickness
                               / (3.0 * bfc * tfc)))
Lp_h = rt_h * math.sqrt(29000.0 / 50.0)
Fyr_s = max(min(0.7 * 50.0, 50.0), 0.5 * 50.0)
Lr_h = math.pi * rt_h * math.sqrt(29000.0 / Fyr_s)
ltb = steel.lateral_torsional_buckling_resistance(
    l_b=L_b, b_fc=bfc, t_fc=tfc, d_c=Dc_nc, t_w=FIELD.web_thickness,
    f_yc=50.0, f_yw=50.0, c_b=1.0, f_bu=f_bu_c)
check("6.10.8.2.3 rt [hand calc]", rt_h, ltb.details["rt"], unit="in")
check("6.10.8.2.3 Lp [hand calc]", Lp_h, ltb.details["Lp"], unit="in")
check("6.10.8.2.3 Lr [hand calc]", Lr_h, ltb.details["Lr"], unit="in")
Fnc_ltb_h = (1.0 - (1.0 - Fyr_s / 50.0) * (L_b - Lp_h) / (Lr_h - Lp_h)) * 50.0
check("6.10.8.2.3 Fnc, inelastic LTB [hand calc]", Fnc_ltb_h, ltb.capacity,
      unit="ksi")

# FLB (6.10.8.2.2): bf/2tf = 8 < 0.38*sqrt(E/Fyc) = 9.15 -> plateau
flb = steel.flange_local_buckling_resistance(
    b_fc=bfc, t_fc=tfc, f_yc=50.0, f_yw=50.0, f_bu=f_bu_c)
check("6.10.8.2.2 Fnc compact-flange plateau = Fyc [hand calc]",
      50.0, flb.capacity, unit="ksi")

# web bend-buckling (6.10.1.9.1)
wbb = steel.web_bend_buckling(
    d_web=FIELD.web_depth, t_w=FIELD.web_thickness, d_c=Dc_nc,
    f_yc=50.0, f_yw=50.0)
k_h = 9.0 / (Dc_nc / FIELD.web_depth) ** 2
Fcrw_h = min(0.9 * 29000.0 * k_h / (FIELD.web_depth
                                    / FIELD.web_thickness) ** 2,
             50.0, 50.0 / 0.7)
check("6.10.1.9.1 Fcrw [hand calc]", Fcrw_h, wbb.capacity, unit="ksi")

# the assembled 6.10.3.2.1 verdict (nonslender web here, so yielding +
# buckling govern; Fcrw reported for completeness)
Fnc = min(ltb.capacity, flb.capacity)
con = steel.constructibility_compression_flange(
    f_bu=f_bu_c, f_l=f_l, f_yc=50.0, f_nc=Fnc, f_crw=wbb.capacity,
    slender_web=False)
print(f"6.10.3.2.1 governing case: {con.details['governing']}, "
      f"ratio = {con.ratio:.2f}")
check("6.10.3.2.1 compression flange passes casting [hand calc]",
      1.0, float(con.ratio >= 1.0))

# tension flange (6.10.3.2.2)
tfl = steel.tension_flange_resistance(f_yt=50.0, f_bu=f_bu_t, f_l=f_l)
check("6.10.3.2.2 tension flange passes casting [hand calc]",
      1.0, float(tfl.ratio >= 1.0))

In [ ]:
# --- casting-stage web shear (6.10.3.3): buckling only, no tension field
V_cast = gam_c * DEMANDS["v_dc1"]        # pier-side shear, wet deck
shr = steel.web_shear_resistance(
    d_web=PIER.web_depth, t_w=PIER.web_thickness, f_yw=50.0,
    v_u=V_cast)                           # unstiffened -> Vn = C*Vp = Vcr
Dtw = PIER.web_depth / PIER.web_thickness
C_h = 1.57 / Dtw ** 2 * (29000.0 * 5.0 / 50.0)   # elastic branch, k = 5
check("6.10.9.3.2-6 shear buckling coefficient C [hand calc]",
      C_h, shr.details["C"])
print(f"Vu = {V_cast:.0f} kip vs Vcr = {shr.capacity:.0f} kip "
      f"(C = {shr.details['C']:.3f})")
check("6.10.3.3 web shear passes casting [hand calc]",
      1.0, float(shr.ratio >= 1.0))

# --- deck tension during staged casting (6.10.3.2.4) --------------------
# When the span-2 positive pour loads a span-1 deck that has already
# cured, the pier-region deck goes into tension on the n-composite
# section.  Representative re-staging moment for the demo sequence
# (hand-entered; a staged model refines it — TODO(midas-api)):
M_stage = -800.0 * 12.0                  # kip-in on the composite section
p_n_pier = girder_neg.props("n")         # uncracked short-term section
y_deck_mid = (PIER.top_flange.thickness + PIER.web_depth
              + PIER.bottom_flange.thickness + DEMO["haunch_in"] + ts / 2)
f_deck = abs(M_stage) * (y_deck_mid - p_n_pier.y_na) / p_n_pier.inertia / n
f_r = 0.24 * math.sqrt(fc)
check("5.4.2.6 fr for the 4.5-ksi deck [published value]",
      0.509, f_r, unit="ksi")
print(f"deck tension {f_deck:.3f} ksi vs phi*fr = {0.9 * f_r:.3f} ksi -> "
      f"{'1% reinforcement REQUIRED (provided in sec. 3)' if f_deck > 0.9 * f_r else 'below cracking threshold'}")
# this staging moment stays below phi*fr, but Service II tension at
# the pier will not — the 6.10.1.7 steel must be there either way
check("6.10.1.7 1% longitudinal steel in place for the pier region [hand calc]",
      1.0, float(A_rs >= 0.01 * A_deck))

## 7. Service II — LRFD 6.10.4

Flange stress limits 0.95·Rh·Fyf (composite) / 0.80·Rh·Fyf, with the
1.3·LL+IM Service II combination.

In [ ]:
# TODO(guide): build f_f from the staged section moduli (DC1 on steel,
# DC2+DW on 3n, LL on n); steel.hybrid_factor(...) if hybrid.
pass

## 8. Fatigue — LRFD 6.10.5 / 6.6.1

Governing details (web-to-flange weld, stiffener welds, shear studs), fatigue
category resistance, and the Fatigue I/II stress ranges from the fatigue
truck.

In [ ]:
# TODO(guide): steel.fatigue_resistance(category=..., n_cycles=...)
# TODO(midas-api): fatigue truck moving-load stress ranges from the model.
pass

## 9. Strength I flexure

Positive flexure: compact composite section check and Dp/Dt ductility
(6.10.7). Negative flexure: FLB / LTB per 6.10.8, or Appendix A6 if the guide
uses it for the compact-web continuous section.

In [ ]:
# TODO(guide):
# steel.compact_composite_positive_flexure(...)
# steel.flange_local_buckling_resistance(...)
# steel.lateral_torsional_buckling_resistance(...)
# steel.tension_flange_resistance(...)
# civilpy.structural.aashto.lrfd.appendix_a6 for the A6 path if used.
pass

## 10. Shear — LRFD 6.10.9

End/interior panel shear with tension-field action, transverse stiffener
spacing, stiffener proportioning (6.10.11.1).

In [ ]:
# TODO(guide):
# steel.web_shear_resistance(...)
# steel.transverse_stiffener_width(...), steel.transverse_stiffener_inertia(...)
pass

## 11. Shear connectors — LRFD 6.10.10

Fatigue pitch governing the layout, strength check of total connectors between
points of max moment and zero moment.

In [ ]:
# TODO(guide):
# steel.shear_connector_strength(...)
# steel.shear_connector_fatigue_pitch(...)
pass

## 12. Bearing stiffeners and Midas cross-check

In [ ]:
# TODO(guide): steel.bearing_stiffener_resistance(...), bearing_stiffener_width(...)

In [ ]:
# TODO(midas-api): "Composite Girder Design Result" is a PSC *design-module* table.
# Verified 2026-07-27: the design result tables (fps, c, Mcr, Av,req,
# FDL/AFDL columns) are NOT exposed through the known /post/TABLE surface —
# they need the PSC Design run configured in the Civil NX UI (design code,
# PSC design parameters, Section Manager rebar) and/or the official JSON
# manual's design TABLE_TYPE names. Analysis-side tables ARE verified:
# BEAMFORCE, BEAMSTRESSPSC (the ten-check-point stress table with
# Sig-Is(shear), Sig-Is(shear+torsion), Sig-Ps(Max/Min) columns), REACTIONG.
print("PENDING: Composite Girder Design Result — needs the PSC Design module (see comment)")

## Validation summary

Every `check()` recorded above, in one table. `PENDING` rows are waiting on
either guide values (hand entry) or the Midas API coming back online.

In [ ]:
summary()